# Configuration & Hardening Audit

This notebook queries the OCP audit SQLite datastore to present findings for:

- **OCP-6 — Platform Usage Guardrails**: Technical guardrails restricting unapproved OpenShift distribution usage (e.g. EKS) where the standard OpenShift Distribution is not used. Reports OCP version, update channel/state, infrastructure platform, control-plane / infrastructure topology, and degraded or unavailable cluster operators.
- **OCP-7 — Policy-as-Code Enforcement**: Use of policy-as-code (e.g. OPA Gatekeeper) to define and enforce cluster-wide governance and compliance rules and centralise policy management. Reports Gatekeeper installation status, ConstraintTemplates, Constraints, enforcement actions, and total violations per constraint.

In [1]:
import os
import sys

import pandas as pd

# Ensure the repo root is importable before loading project modules.
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from notebook_style import bootstrap, style_table  # noqa: E402
from sqlalchemy import func  # noqa: E402

print("python:", sys.executable)
print("cwd:", os.getcwd())

# bootstrap() adds ../datastore to sys.path, which is required before
# importing schema.models below.
session, engine = bootstrap()

from schema.models import (  # noqa: E402
    Cluster,
    PlatformGuardrail,
    PolicyAsCodeConstraint,
)

print(f"Connected to: {engine.url}")

python: /home/vagrant/git/openshift-csv-exporter/notebook/.venv/bin/python
cwd: /home/vagrant/git/openshift-csv-exporter/notebook
Connected to: sqlite:////home/vagrant/git/openshift-csv-exporter/datastore/ocp_audit.db


## Cluster Inventory

In [2]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
style_table(df_clusters)

9 cluster(s) in dataset


---
## OCP-6: Platform Usage Guardrails

Per-cluster posture: OCP version, infrastructure platform, control-plane / infrastructure topology, update channel and state, and the number of cluster operators currently degraded or unavailable. Clusters that drift from the approved OpenShift distribution baseline (wrong platform, non-HA topology in production, off-channel versions, or degraded operators) should be investigated.

In [3]:
df_guardrails = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        PlatformGuardrail.ocp_version,
        PlatformGuardrail.platform,
        PlatformGuardrail.control_plane_topology,
        PlatformGuardrail.infrastructure_topology,
        PlatformGuardrail.update_channel,
        PlatformGuardrail.update_state,
        PlatformGuardrail.total_operators,
        PlatformGuardrail.degraded_count,
        PlatformGuardrail.unavailable_count,
    )
    .join(Cluster, PlatformGuardrail.cluster_id == Cluster.id)
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)
print(f"Platform guardrail posture: {len(df_guardrails)} cluster(s)")
style_table(df_guardrails)

Platform guardrail posture: 9 cluster(s)


### OCP-6: Degraded / Unavailable Operators

Clusters with one or more cluster operators in a `Degraded=True` or `Available=False` state. The `degraded_operators` and `unavailable_operators` fields are `;`-separated lists of operator names.

In [4]:
df_op_issues = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        PlatformGuardrail.degraded_count,
        PlatformGuardrail.degraded_operators,
        PlatformGuardrail.unavailable_count,
        PlatformGuardrail.unavailable_operators,
    )
    .join(Cluster, PlatformGuardrail.cluster_id == Cluster.id)
    .filter(
        (PlatformGuardrail.degraded_count > 0)
        | (PlatformGuardrail.unavailable_count > 0)
    )
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)
if df_op_issues.empty:
    print("No degraded or unavailable cluster operators across all clusters.")
else:
    print(f"{len(df_op_issues)} cluster(s) with degraded or unavailable operators")
style_table(df_op_issues)

3 cluster(s) with degraded or unavailable operators


### OCP-6: Compliance Flags

Clusters that fail one or more platform-guardrail hardening checks:

- **Non-HA control plane** — `control_plane_topology` is not `HighlyAvailable` (e.g. `SingleReplica`)
- **Update not completed** — `update_state` is not `Completed`
- **Operators degraded** — at least one cluster operator reports `Degraded=True`
- **Operators unavailable** — at least one cluster operator reports `Available=False`

In [5]:
df_g = df_guardrails.copy()


def _s(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return ""
    return str(val).strip()


def _non_ha(row):
    return _s(row.get("control_plane_topology")) not in ("", "HighlyAvailable")


def _update_incomplete(row):
    state = _s(row.get("update_state"))
    return state != "" and state != "Completed"


def _gt_zero(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return False
    try:
        return int(val) > 0
    except (TypeError, ValueError):
        return False


df_g["non_ha_control_plane"] = df_g.apply(_non_ha, axis=1)
df_g["update_incomplete"] = df_g.apply(_update_incomplete, axis=1)
df_g["operators_degraded"] = df_g["degraded_count"].apply(_gt_zero)
df_g["operators_unavailable"] = df_g["unavailable_count"].apply(_gt_zero)

flag_cols = [
    "non_ha_control_plane",
    "update_incomplete",
    "operators_degraded",
    "operators_unavailable",
]
df_guard_flags = df_g[df_g[flag_cols].any(axis=1)][
    [
        "cluster_name",
        "ocp_version",
        "platform",
        "control_plane_topology",
        "update_state",
        "degraded_count",
        "unavailable_count",
        *flag_cols,
    ]
]

print(f"{len(df_guard_flags)} cluster(s) fail at least one OCP-6 hardening check")
style_table(df_guard_flags)

3 cluster(s) fail at least one OCP-6 hardening check


---
## OCP-7: Policy-as-Code Enforcement

Per-cluster summary of OPA Gatekeeper installation status, the number of `ConstraintTemplate`s and `Constraint`s defined, and the total violations observed across all constraints. A cluster is non-compliant if Gatekeeper is not installed or no constraints are defined.

In [6]:
df_pac_summary = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        func.max(PolicyAsCodeConstraint.gatekeeper_installed).label(
            "gatekeeper_installed"
        ),
        func.max(PolicyAsCodeConstraint.gatekeeper_namespace).label(
            "gatekeeper_namespace"
        ),
        func.count(func.distinct(PolicyAsCodeConstraint.constraint_template)).label(
            "templates"
        ),
        func.count(func.distinct(PolicyAsCodeConstraint.constraint_name)).label(
            "constraints"
        ),
        func.coalesce(func.sum(PolicyAsCodeConstraint.total_violations), 0).label(
            "total_violations"
        ),
    )
    .join(Cluster, PolicyAsCodeConstraint.cluster_id == Cluster.id)
    .group_by(Cluster.cluster_name)
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)
print(f"Policy-as-code posture: {len(df_pac_summary)} cluster(s)")
style_table(df_pac_summary)

Policy-as-code posture: 9 cluster(s)


### OCP-7: Constraint Inventory

Every Gatekeeper `Constraint` defined across all clusters, with its enforcement action, current violation count, and which Kubernetes kinds and namespaces it matches.

In [7]:
df_constraints = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        PolicyAsCodeConstraint.constraint_template,
        PolicyAsCodeConstraint.constraint_name,
        PolicyAsCodeConstraint.enforcement_action,
        PolicyAsCodeConstraint.total_violations,
        PolicyAsCodeConstraint.match_kinds,
        PolicyAsCodeConstraint.match_namespaces,
    )
    .join(Cluster, PolicyAsCodeConstraint.cluster_id == Cluster.id)
    .filter(PolicyAsCodeConstraint.constraint_name.isnot(None))
    .order_by(
        Cluster.cluster_name,
        PolicyAsCodeConstraint.constraint_template,
        PolicyAsCodeConstraint.constraint_name,
    )
    .statement,
    engine,
)
print(f"{len(df_constraints)} constraint(s) defined")
style_table(df_constraints)

15 constraint(s) defined


### OCP-7: Constraints with Active Violations

Constraints with `total_violations > 0`. Constraints in `dryrun` or `warn` mode allow non-compliant resources through; only `deny` actually blocks them.

In [8]:
df_violations = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        PolicyAsCodeConstraint.constraint_template,
        PolicyAsCodeConstraint.constraint_name,
        PolicyAsCodeConstraint.enforcement_action,
        PolicyAsCodeConstraint.total_violations,
        PolicyAsCodeConstraint.match_kinds,
    )
    .join(Cluster, PolicyAsCodeConstraint.cluster_id == Cluster.id)
    .filter(PolicyAsCodeConstraint.total_violations > 0)
    .order_by(
        PolicyAsCodeConstraint.total_violations.desc(),
        Cluster.cluster_name,
    )
    .statement,
    engine,
)
if df_violations.empty:
    print("No active constraint violations across all clusters.")
else:
    print(f"{len(df_violations)} constraint(s) with active violations")
style_table(df_violations)

12 constraint(s) with active violations


### OCP-7: Compliance Flags

Clusters that fail one or more policy-as-code hardening checks:

- **Gatekeeper not installed** — no `openshift-gatekeeper-system` or `gatekeeper-system` namespace detected
- **No constraints defined** — Gatekeeper is installed but no `Constraint` resources are configured
- **Non-enforcing constraints** — at least one constraint is in `dryrun` or `warn` mode (does not block non-compliant resources)
- **Active violations** — at least one constraint reports `total_violations > 0`

In [9]:
all_clusters = pd.read_sql(
    session.query(Cluster.cluster_name).statement,
    engine,
)

df_pac_full = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        PolicyAsCodeConstraint.gatekeeper_installed,
        PolicyAsCodeConstraint.constraint_name,
        PolicyAsCodeConstraint.enforcement_action,
        PolicyAsCodeConstraint.total_violations,
    )
    .join(Cluster, PolicyAsCodeConstraint.cluster_id == Cluster.id)
    .statement,
    engine,
)

rows = []
for cluster_name in sorted(all_clusters["cluster_name"].unique()):
    sub = df_pac_full[df_pac_full["cluster_name"] == cluster_name]
    if sub.empty:
        gk_installed = False
        has_constraints = False
        non_enforcing = False
        active_violations = False
    else:
        gk_installed = bool(sub["gatekeeper_installed"].fillna(False).any())
        constraint_rows = sub[
            sub["constraint_name"].notna() & (sub["constraint_name"] != "")
        ]
        has_constraints = not constraint_rows.empty
        non_enforcing = bool(
            constraint_rows["enforcement_action"]
            .fillna("")
            .str.lower()
            .isin(["dryrun", "warn"])
            .any()
        )
        active_violations = bool(
            (constraint_rows["total_violations"].fillna(0) > 0).any()
        )

    rows.append(
        {
            "cluster_name": cluster_name,
            "gatekeeper_installed": gk_installed,
            "gatekeeper_not_installed": not gk_installed,
            "no_constraints_defined": gk_installed and not has_constraints,
            "non_enforcing_constraints": non_enforcing,
            "active_violations": active_violations,
        }
    )

df_pac_flags_all = pd.DataFrame(rows)
flag_cols = [
    "gatekeeper_not_installed",
    "no_constraints_defined",
    "non_enforcing_constraints",
    "active_violations",
]
df_pac_flags = df_pac_flags_all[df_pac_flags_all[flag_cols].any(axis=1)]

print(f"{len(df_pac_flags)} cluster(s) fail at least one OCP-7 hardening check")
style_table(df_pac_flags)

9 cluster(s) fail at least one OCP-7 hardening check
